# ScannerAgent & MessengerAgent

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [ ]:
# Importing Libraries

import os
import requests
import logging
from ollama import chat
from agents.deals import ScrapedDeal, DealSelection

MODEL = "qwen3"

In [ ]:
deals = ScrapedDeal.fetch(show_progress=True)

In [ ]:
len(deals)

In [ ]:
print(deals[10].describe())

### We are going to ask Qwen to summarize deals and identify their price

In [ ]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scrapped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scrapped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [ ]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [ ]:
# This is the json schema the model will return its results

DealSelection.model_json_schema()

In [ ]:
response = chat(
    model=MODEL,
    messages=messages,
    format=DealSelection.model_json_schema(),
    think=False,
    options={
        "temperature": 0,
    },
)

results = DealSelection.model_validate_json(
    response.message.content
)

print(results)

In [ ]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.scanner_agent import ScannerAgent

agent = ScannerAgent()
result = agent.scan()

In [ ]:
result

### Messaging Agent Will be Added Later